In [2]:
import argparse
import pandas as pd
import numpy as np
from preprocess import preprocess
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_recall_curve, precision_score, recall_score, average_precision_score, roc_curve, auc, confusion_matrix, mean_squared_error,classification_report
import time

import matplotlib.pyplot as plt
from keras.utils import to_categorical

trainset = pd.read_csv('./data/NSL-KDD/KDDTrain+.txt', sep=",", header=None)
testset = pd.read_csv('./data/NSL-KDD/KDDTest+.txt', sep=",", header=None)

#下面部分是GAN训练并且生成数据
processor = preprocess()
print("数据预处理....")
df_train, df_test, train_Normal, train_R2L, train_U2R, train_Dos, train_Probe,test_Normal, test_R2L, test_U2R, test_Dos, test_Probe,train_Attack,test_Attack = processor.create_df(df_train=trainset, df_test=testset)
# normal_df, R2L_df, R2L_df_train, R2L_df_test = processor.create_df(df_train=trainset, df_test=testset)
print("已完成数据预处理")


Using TensorFlow backend.


数据预处理....
已完成数据预处理


In [3]:
dt={'Dos':pd.Series([len(train_Dos),len(test_Dos)],index=['Train','Test']),
   'Probe':pd.Series([len(train_Probe),len(test_Probe)],index=['Train','Test']),
   'R2L':pd.Series([len(train_R2L),len(test_R2L)],index=['Train','Test']),
   'U2R':pd.Series([len(train_U2R),len(test_U2R)],index=['Train','Test']),
   'Normal':pd.Series([len(train_Normal),len(test_Normal)],index=['Train','Test']),
   'Total_attack':pd.Series([len(train_Attack),len(test_Attack)],index=['Train','Test']),
   'Total':pd.Series([len(df_train),len(df_test)],index=['Train','Test'])}
type_df=pd.DataFrame(dt)
cols = ['Dos','Probe','R2L','U2R','Normal','Total_attack','Total']
type_df = type_df[cols]
display(type_df)


,Dos,Probe,R2L,U2R,Normal,Total_attack,Total
Train,11656,45927,995,52,67343,58630,125973
Test,2421,7460,2885,67,9711,12833,22544


In [3]:
#加入生成数据的R2L类别二分类
generated_data = pd.read_csv('./output/fake_examples.csv', sep=",", header=None)
generated_data = processor.gererated_preprocess(generated_data)

#train_add_BiR2L = train_Normal.append(generated_data)

#train_add_BiR2L = processor.merge_df(df_train, generated_data)
#train_add_BiR2L = np.concatenate(df_train, generated_data)

train_BiR2L = train_Normal.append(train_R2L)
train_add_BiR2L = train_BiR2L.append(generated_data)
X_train_add_R2L,y_train_add_R2L = processor.split_df(train_add_BiR2L)

test_BiR2L = test_Normal.append(test_R2L)
X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)


In [4]:
#Dos类二分类
train_BiDos = train_Normal.append(train_Dos)
X_train_Dos,y_train_Dos = processor.split_df(train_BiDos)
y_train_Dos = y_train_Dos.replace(2,1)

test_BiDos = test_Normal.append(test_Dos)
X_test_Dos,y_test_Dos = processor.split_df(test_BiDos)
y_test_Dos = y_test_Dos.replace(2,1)

In [5]:
#R2L类二分类
#train_BiR2L = train_Normal.append(train_R2L)
X_train_R2L,y_train_R2L = processor.split_df(train_BiR2L)


# test_BiR2L = test_Normal.append(test_R2L)
# X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)


In [6]:
#U2R类二分类
train_BiU2R = train_Normal.append(train_U2R)
X_train_U2R,y_train_U2R = processor.split_df(train_BiU2R)
y_train_U2R = y_train_U2R.replace(4,1)

test_BiU2R = test_Normal.append(test_U2R)
X_test_U2R,y_test_U2R = processor.split_df(test_BiU2R)
y_test_U2R = y_test_U2R.replace(4,1)

In [7]:
#Probe类二分类
train_BiProbe = train_Normal.append(train_Probe)
X_train_Probe,y_train_Probe = processor.split_df(train_BiProbe)
y_train_Probe = y_train_Probe.replace(3,1)

test_BiProbe = test_Normal.append(test_Probe)
X_test_Probe,y_test_Probe = processor.split_df(test_BiProbe)
y_test_Probe = y_test_Probe.replace(3,1)

In [8]:
#全局二分类

X_train_Bi,y_train_Bi = processor.split_df(df_train)
y_train_Bi = y_train_Bi.replace(4,1).replace(3,1).replace(2,1)

X_test_Bi,y_test_Bi = processor.split_df(df_test)
y_test_Bi = y_test_Bi.replace(4,1).replace(3,1).replace(2,1)


In [9]:
#Y = np.concatenate([y_genuine, y_imposite], axis=0)

In [10]:
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.layers import LSTM, SimpleRNN, GRU
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
from sklearn.preprocessing import OneHotEncoder



def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=41,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

def DNNBinary(X_train,X_test,y_train,y_test):
#     y_train = np.array(y_train).astype(np.int)
#     y_test = np.array(y_test).astype(np.int)
#     y_train = to_categorical(y_train).astype(np.int)
#     y_test = to_categorical(y_test).astype(np.int)
#     print(y_test.shape)
    
#     enc = OneHotEncoder(sparse = False)
#     y_test = enc.fit_transform(y_test.reshape(len(y_test),1))
#     y_train = enc.fit_transform(y_train.reshape(len(y_train),1))
    
    batch_size = 64
    checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
    csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
    #model = KerasClassifier(build_fn=build_model, epochs=2, batch_size=batch_size)
    model = KerasClassifier(build_fn=build_model, epochs=2, batch_size=batch_size)
    model.fit(X_train, y_train, validation_data=(X_test, y_test),batch_size=batch_size, epochs=10, callbacks=[checkpointer,csv_logger])
    #model.save("DNNResult/dnn1layer_model.hdf5")
    
    y_pred = model.predict(X_test)  # 预测结果
    y_ture = y_test
    y_pred = y_pred.flatten()

    acc = np.sum(y_pred==y_ture)/len(y_ture)
    print("准确率（Accuracy）：f",acc)
    pre = precision_score(y_ture,y_pred)
    print("精确率(Precision)：",pre)
    rec = recall_score(y_ture,y_pred)
    print("召回率(灵敏度/真阳率/Sensitivity/Recall/TPR)：",rec)
    f_1 = 2*pre*rec/(pre+rec)
    print("F_1 score:",f_1)
    print("特异度：真实值是Negative的所有结果中，模型预测对的比重")
    #acu_curve(y_ture,y_pred)
    return acc, pre, rec, f_1

#     accuracy = cross_val_score(model, X_test, y_test, cv=2, scoring='accuracy')
#     print("Accuracy: %0.5f (+/- %0.5f)" % (accuracy.mean(), accuracy.std() * 2))
#     precision = cross_val_score(model, X_test, y_test, cv=2, scoring='precision')
#     print("Precision: %0.5f (+/- %0.5f)" % (precision.mean(), precision.std() * 2))
#     recall = cross_val_score(model, X_test, y_test, cv=2, scoring='recall')
#     print("Recall: %0.5f (+/- %0.5f)" % (recall.mean(), recall.std() * 2))
#     f = cross_val_score(model, X_test, y_test, cv=2, scoring='f1')
#     print("F-measure: %0.5f (+/- %0.5f)" % (f.mean(), f.std() * 2))

In [11]:
    #41指的是列标为41的那一列数据
# y_test
# len(y_test)
#len(X_test)

NameError: name 'y_test' is not defined

In [12]:


def decisionTree(X_train,X_test,y_train,y_test):
    #实现决策树算法

    print("开始训练决策树...")
    # print(X_train["Flag"])
    # print(R2L_df.shape[0], R2L_df_train.shape[0], R2L_df_test.shape[0], X_train.shape[0])
    #clf_R2L = DecisionTreeClassifier(criterion="entropy")
    clf_R2L = DecisionTreeClassifier(random_state=0)
    clf_R2L.fit(X_train, y_train.astype('int'))
    """训练DT;在sklearn 模型训练是出现如下报错：‘ValueError: Unknown label type: ‘unknown’’该怎么解决？
    以GBDT为例：train_y后加上astype(‘int’)即可"""
    y_pred = clf_R2L.predict(X_test)  # 预测结果
    # Create confusion matrix
    y_ture = y_test
    print("——————实验结果————")
    #print("accuracy:",accuracy_score(y_ture, y_pred))
    acc = np.sum(y_pred==y_ture)/len(y_ture)
    pre = precision_score(y_ture,y_pred)
    rec = recall_score(y_ture,y_pred)
    f_1 = 2*pre*rec/(pre+rec)
    print("准确率（Accuracy）：",acc)
    print("精确率(Precision)：",pre)
    print("召回率(灵敏度/真阳率/Sensitivity/Recall/TPR)：",rec)
    print("F_1 score:",f_1)
    print("特异度：真实值是Negative的所有结果中，模型预测对的比重")
    #acu_curve(y_ture,y_pred)
    return acc, pre, rec, f_1

def acu_curve(y,prob):
#绘制ROC曲线，y真实、prob预测
    fpr,tpr,threshold = roc_curve(y,prob) ###计算真阳性率和假阳性率
    print("fpr,tpr,threshold:",fpr,tpr,threshold)
    roc_auc = auc(fpr,tpr) ###计算auc的值
    lw = 2
    plt.figure(figsize=(10,10))
    plt.plot(fpr, tpr, color='darkorange',lw=lw, label='ROC curve (AUC = %0.3f)' % roc_auc) ###假正率为横坐标，真正率为纵坐标做曲线
    plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC curve and AUC value')
    plt.legend(loc="lower right")
    plt.show()

    #绘制折线图
    plt.figure(2)
    showPlot(y,prob)

def showPlot(test_y,test_pred):
#绘制折线图

    plt.rcParams["font.sans-serif"]=["SimHei"]
    plt.title("真实分类和测试类的折线图")
    plt.xlabel("数量")
    plt.ylabel("分类")
    plt.plot(range(len(test_y)),test_y,"r")
    plt.plot(range(len(test_pred)),test_pred,"g")
    plt.legend(["真实","测试"])
    plt.show()



In [13]:
#y_train.replace(np.nan, 0, inplace=True)
#y_train.replace(np.inf, 0, inplace=True)


#X_train.replace(np.nan, 0, inplace=True)
#X_train.replace(np.inf, 0, inplace=True)
# y_train.replace(np.nan, 0, inplace=True)
# y_train.replace(np.inf, 0, inplace=True)
# y_train.replace(np.nan, 0, inplace=True)
# y_train.replace(np.inf, 0, inplace=True)

In [14]:
X_train.shape
y_test.shape

NameError: name 'X_train' is not defined

In [15]:
#R2L二分类
time_start=time.time()
acc_R2L_DT, pre_R2L_DT, rec_R2L_DT, f1_R2L_DT = decisionTree(X_train_R2L,X_test_R2L,y_train_R2L,y_test_R2L)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

time_start=time.time()
acc_R2L_DNNBi, pre_R2L_DNNBi, rec_R2L_DNNBi, f1_R2L_DNNBi = DNNBinary(X_train_R2L,X_test_R2L,y_train_R2L,y_test_R2L)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

####################################
#Dos二分类
time_start=time.time()
acc_Dos_DT, pre_Dos_DT, rec_Dos_DT, f1_Dos_DT = decisionTree(X_train_Dos,X_test_BiDos,y_train_BiDos,y_test_BiDos)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

time_start=time.time()
acc_Dos_DNNBi, pre_Dos_DNNBi, rec_Dos_DNNBi, f1_Dos_DNNBi = DNNBinary(X_train_Dos,X_test_Dos,y_train_Dos,y_test_Dos)
time_end=time.time()
print('耗时(s)：',time_end-time_start)
####################################
#U2R二分类
time_start=time.time()
acc_U2R_DT, pre_U2R_DT, rec_U2R_DT, f1_U2R_DT = decisionTree(X_train_U2R,X_test_U2R,y_train_U2R,y_test_U2R)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

time_start=time.time()
acc_U2R_DNNBi, pre_U2R_DNNBi, rec_U2R_DNNBi, f1_U2R_DNNBi = DNNBinary(X_train_U2R,X_test_U2R,y_train_U2R,y_test_U2R)
time_end=time.time()
print('耗时(s)：',time_end-time_start)
####################################
#Probe二分类
time_start=time.time()
acc_Probe_DT, pre_Probe_DT, rec_Probe_DT, f1_Probe_DT = decisionTree(X_train_Probe,X_test_Probe,y_train_Probe,y_test_Probe)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

time_start=time.time()
acc_Probe_DNNBi, pre_Probe_DNNBi, rec_Probe_DNNBi, f1_Probe_DNNBi = DNNBinary(X_train_Probe,X_test_Probe,y_train_Probe,y_test_Probe)
time_end=time.time()
print('耗时(s)：',time_end-time_start)
####################################
#无加入全局二分类
time_start=time.time()
acc_Bi_DT, pre_Bi_DT, rec_Bi_DT, f1_Bi_DT = decisionTree(X_train_Bi,X_test_Bi,y_train_Bi,y_test_Bi)
time_end=time.time()
print('耗时(s)：',time_end-time_start)

time_start=time.time()
acc_Bi_DNNBi, pre_Bi_DNNBi, rec_Bi_DNNBi, f1_Bi_DNNBi = DNNBinary(X_train_Bi,X_test_Bi,y_train_Bi,y_test_Bi)
time_end=time.time()
print('耗时(s)：',time_end-time_start)










开始训练决策树...
——————实验结果————
准确率（Accuracy）： 0.798507462687
精确率(Precision)： 0.728590250329
召回率(灵敏度/真阳率/Sensitivity/Recall/TPR)： 0.191681109185
F_1 score: 0.303512623491
特异度：真实值是Negative的所有结果中，模型预测对的比重
耗时(s)： 0.4063756465911865
Train on 68338 samples, validate on 12596 samples
Epoch 1/10
68338/68338 [==============================] - 5s 80us/step - loss: 0.0370 - acc: 0.9879 - val_loss: 1.8226 - val_acc: 0.7710

Epoch 00001: loss improved from inf to 0.03698, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/10
68338/68338 [==============================] - 4s 56us/step - loss: 0.0190 - acc: 0.9935 - val_loss: 1.8239 - val_acc: 0.7719

Epoch 00002: loss improved from 0.03698 to 0.01903, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/10
68338/68338 [==============================] - 4s 51us/step - loss: 0.0161 - acc: 0.9945 - val_loss: 2.0159 - val_acc: 0.7711

Epoch 00003: loss improved from 0.01903 to 0.01611, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/10
68338/683

NameError: name 'X_test_BiDos' is not defined

In [74]:
dt_DT={'acc':pd.Series([acc_Dos_DT,acc_Probe_DT,acc_U2R_DT,acc_R2L_DT,acc_Bi_DT],index=['Dos','Probe','U2R','R2L','all_Bi']),
   'pre':pd.Series([pre_Dos_DT,pre_Probe_DT,pre_U2R_DT,pre_R2L_DT,pre_Bi_DT],index=['Dos','Probe','U2R','R2L','all_Bi']),
   'rec':pd.Series([rec_Dos_DT,rec_Probe_DT,rec_U2R_DT,rec_R2L_DT,rec_Bi_DT],index=['Dos','Probe','U2R','R2L','all_Bi']),
   'f1':pd.Series([f1_Dos_DT,f1_Probe_DT,f1_U2R_DT,f1_R2L_DT,f1_Bi_DT],index=['Dos','Probe','U2R','R2L','all_Bi'])}
type_df_DT=pd.DataFrame(dt_DT)
cols_DT = ['acc','pre','rec','f1']
type_df_DT = type_df_DT[cols_DT]
type_df_DT.round(4)
#display(type_df_DT)



,acc,pre,rec,f1
Dos,0.9129,0.8451,0.6898,0.7596
Probe,0.9104,0.9780,0.8119,0.8873
U2R,0.9920,0.2609,0.0896,0.1333
R2L,0.7985,0.7286,0.1917,0.3035
all_Bi,0.8759,0.6629,0.7699,0.7124


In [75]:
dt_DNNBi={'acc':pd.Series([acc_Dos_DNNBi,acc_Probe_DNNBi,acc_U2R_DNNBi,acc_R2L_DNNBi,acc_Bi_DNNBi],index=['Dos','Probe','U2R','R2L','all_Bi']),
   'pre':pd.Series([pre_Dos_DNNBi,pre_Probe_DNNBi,pre_U2R_DNNBi,pre_R2L_DNNBi,pre_Bi_DNNBi],index=['Dos','Probe','U2R','R2L','all_Bi']),
   'rec':pd.Series([rec_Dos_DNNBi,rec_Probe_DNNBi,rec_U2R_DNNBi,rec_R2L_DNNBi,rec_Bi_DNNBi],index=['Dos','Probe','U2R','R2L','all_Bi']),
    'f1':pd.Series([f1_Dos_DNNBi,f1_Probe_DNNBi,f1_U2R_DNNBi,f1_R2L_DNNBi,f1_Bi_DNNBi],index=['Dos','Probe','U2R','R2L','all_Bi'])}
type_df_DNNBi=pd.DataFrame(dt_DNNBi)
cols_DNNBi = ['acc','pre','rec','f1']
type_df_DNNBi = type_df_DNNBi[cols_DNNBi]
type_df_DNNBi.round(4)

,acc,pre,rec,f1
Dos,0.9032,0.8875,0.5898,0.7087
Probe,0.9266,0.9865,0.8425,0.9088
U2R,0.9940,0.7500,0.1791,0.2892
R2L,0.7833,0.9058,0.0600,0.1125
all_Bi,0.9315,0.8729,0.7687,0.8175
